# Azul Rocket — run everything from the GitHub repo

[github.com/laAzulNanotec/azul-rocket](https://github.com/laAzulNanotec/azul-rocket)

This notebook clones the **entire** repo into Colab, then runs the solar optimizer and plot generators directly from the cloned files.

Why clone everything? So you can also browse the MATLAB scripts, the chapter docx, the CAD outputs, and the reference images — all in one place, all available to your code, all re-pullable with one command.

**Just run the cells in order.**

## Setup — clone repo + install dependencies

In [ ]:
# Clone the repo (or pull latest if already cloned)
import os
REPO = '/content/azul-rocket'

if not os.path.exists(REPO):
    !git clone https://github.com/laAzulNanotec/azul-rocket.git {REPO}
else:
    !cd {REPO} && git pull --rebase

!pip install -q -r {REPO}/requirements.txt

In [ ]:
# Switch into the repo and add it to the import path
import os, sys
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'python'))

from solar_optimizer import (
    run_full_study, plot_all_dome_shapes, plot_hourly_harvest,
    plot_shape_comparison, plot_site_comparison, plot_3d_dome,
    hourly_harvest, compute_geometry, SITES, SHAPES,
)

print('Working dir:    ', os.getcwd())
print('Available sites: ', list(SITES.keys()))
print('Available shapes:', list(SHAPES.keys()))

## Quickest path — one line, full study

Runs the optimizer, generates all five reference figures, saves them into `azul-rocket/images/`, and shows them inline.

In [ ]:
h = run_full_study(
    site='peten', width_ft=7.5, floor_h_ft=4.0,
    n_long=4, n_row=2, n_north=2, derate=0.87,
    save_dir='images',   # relative to repo root
    show=True,
)

## Or run via the command-line entry point

Equivalent to running `python run.py` from your terminal.

In [ ]:
!python run.py --site peten --no-show

## Sweep across all four sites

In [ ]:
!python run.py --all-sites --no-show
!ls images/

## Individual plots — explore one at a time

In [ ]:
# All five vault shapes side by side
plot_all_dome_shapes(width_ft=7.5, floor_h_ft=4.0)

In [ ]:
# Hourly harvest curve for the chapter spec
fig, h = plot_hourly_harvest(
    site='peten', shape='low120',
    n_long=4, n_row=2, n_north=2, derate=0.87,
)
print(f"Daily: {h['E_day']:.1f} kWh - Annual: {h['E_year']:.2f} MWh - Peak: {h['P_peak_arc']:.2f} kW")

In [ ]:
# Shape comparison at each site
for site in ['peten', 'kohala', 'austin', 'california']:
    plot_shape_comparison(site=site)

In [ ]:
# 3D rendering for each vault shape
for shape in ['semi180', 'low120', 'raised240', 'gothic', 'catenary']:
    plot_3d_dome(shape=shape, width_ft=7.5, floor_h_ft=4.0, length_ft=24)

## Custom site — add your own

In [ ]:
SITES['atacama'] = {
    'name': 'Atacama, Chile',
    'lat': -24.0, 'psh': 7.5, 'albedo': 0.55, 'canyon': 80,
}

plot_shape_comparison(site='atacama')
plot_hourly_harvest(site='atacama', shape='low120')

## Browse the repo — see what else is here

In [ ]:
# Show the whole tree
!find . -type f -not -path './.git*' -not -path './__pycache__*' | sort

In [ ]:
# Read the chapter prose (header)
!head -40 README.md

## Download the generated plots

Zips `images/` and triggers a browser download.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/chapter12_plots', 'zip', 'images')
files.download(zip_path)

## Push your variant back to the repo (optional)

If you fork the repo and want to contribute your own site or vault variant:

```bash
# In a Colab cell or terminal, after editing
!cd azul-rocket && git config user.name "Your Name"
!cd azul-rocket && git config user.email "you@example.com"
!cd azul-rocket && git add . && git commit -m "Add Atacama site"
!cd azul-rocket && git push
```

Or open a PR through the GitHub web UI with your modified `python/solar_optimizer.py`.